In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import optuna


In [28]:
df = pd.read_csv("preprocess/features_raw.csv", parse_dates=["Date"])

In [29]:
df.head(5)

,Date,Country,interest_differential,real_interest_rate,fx_depreciation_mom,reserves_change_mom,reserves_change_yoy,reserve_adequacy_months,m2_reserves_ratio,inflation_accel_3m,...,Inflation_YoY_%_lag6,fx_depreciation_mom_lag1,fx_depreciation_mom_lag3,fx_depreciation_mom_lag6,reer_misalignment_lag1,reer_misalignment_lag3,reer_misalignment_lag6,capital_flow_pct_lag1,capital_flow_pct_lag3,capital_flow_pct_lag6
0,1974-01-01,Brazil,8.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974-02-01,Brazil,9.03,NaN,NaN,3.198559,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1974-03-01,Brazil,8.65,NaN,NaN,7.182787,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1974-04-01,Brazil,7.49,NaN,NaN,-0.167378,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1974-05-01,Brazil,6.69,NaN,NaN,-2.864064,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
df.groupby("Country").size() 

Country
Brazil            625
Chile             624
China             577
Colombia          625
Czech Republic    409
Greece            624
Hungary           541
India             624
Indonesia         624
Kuwait            624
Malaysia          625
Mexico            624
Peru              625
Philippines       624
Poland            553
Qatar             622
Saudi Arabia      624
South Africa      625
South Korea       625
Thailand          624
Turkey            625
UAE               624
dtype: int64

In [9]:
df.columns

Index(['Date', 'Country', 'interest_differential', 'real_interest_rate',
       'fx_depreciation_mom', 'reserves_change_mom', 'reserves_change_yoy',
       'reserve_adequacy_months', 'm2_reserves_ratio', 'inflation_accel_3m',
       'Inflation_YoY_%', 'trade_to_reserves', 'export_growth_yoy',
       'm2_growth_yoy', 'credit_boom_yoy', 'reer_misalignment',
       'std_vulnerability', 'reserves_to_std_proxy', 'capital_flow_pct',
       'reserve_adequacy_months_diff', 'm2_reserves_ratio_diff',
       'Inflation_YoY_%_diff', 'trade_to_reserves_diff', 'm2_growth_yoy_diff',
       'reer_misalignment_diff', 'std_vulnerability_diff',
       'reserves_to_std_proxy_diff', 'interest_differential_log',
       'Inflation_YoY_%_log', 'inflation_accel_3m_log', 'emp_index',
       'emp_crisis', 'sudden_stop', 'crisis_in_3m', 'crisis_in_6m',
       'crisis_in_12m', 'interest_differential_lag1',
       'interest_differential_lag3', 'interest_differential_lag6',
       'reserves_change_mom_lag1', 'reserv

## Step 1: Panel data preparation (Date–Country)

- The raw `features_raw` file is in **long format**: each row is one `(Date, Country)` observation (for example `1985-01-01 / Brazil / features...`, later `1985-01-01 / Philippines / features...`).

- We **do not merge** different countries with the same date into a single row; instead, each country–date pair is its own training sample.

- We set `Date` as the index (while keeping `Country` as a column), drop the crisis label columns and `Country` from the feature set, and use all remaining numeric columns as features.

- This lets the models learn on a stacked panel of all countries over time, while we still control which countries go into training, validation, and the held-out backtest (Philippines) using the `Country` column and the time-based CV plan.


In [ ]:
# Step 1: Data preparation for panel (Date, Country) dataset
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 1) Make sure Date is datetime and sort panel
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Country"])
df = df.set_index("Date")  # use Date as index, keep Country column

# 2) Define targets and feature columns
targets = ["crisis_in_3m", "crisis_in_6m", "crisis_in_12m"]
feature_cols = [c for c in df.columns if c not in targets + ["Country"]]

# 3) Build feature matrix X and target dictionaries (long format: one row per Date–Country)
X = df[feature_cols].fillna(0)   # raw, zero-filled features
y_dict = {t: df[t] for t in targets}

print("Data preparation complete.")
print(f"Total observations (Date–Country rows): {len(df)}")
print(f"Number of features: {len(feature_cols)}")

Data preparation complete. Ready for modeling.


In [ ]:
# Step 2: Load CV fold plan and define backtest country
cv_plan = pd.read_csv("preprocess/cv_fold_plan.csv", parse_dates=["train_start", "train_end", "test_start", "test_end"])

backtest_country = "Philippines"


cv_plan.head()

In [ ]:
# Step 3: Train and Evaluate Models for crisis_in_6m
target = "crisis_in_6m"
results = {}

for fold, row in cv_plan.iterrows():
    # Time-based masks excluding backtest country
    train_mask = (df.index >= row["train_start"]) & (df.index <= row["train_end"]) & (df["Country"] != backtest_country)
    test_mask = (df.index >= row["test_start"]) & (df.index <= row["test_end"]) & (df["Country"] != backtest_country)
    
    # Raw features (zero-filled) for tree models
    X_train = df.loc[train_mask, feature_cols].fillna(0)
    y_train = df.loc[train_mask, target]
    X_test = df.loc[test_mask, feature_cols].fillna(0)
    y_test = df.loc[test_mask, target]
    
    # Scaled features only for Logistic Regression
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Logistic Regression (scaled)
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train_scaled, y_train)
    y_pred_lr = lr.predict(X_test_scaled)
    y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]
    acc_lr = accuracy_score(y_test, y_pred_lr)
    auc_lr = roc_auc_score(y_test, y_proba_lr)
    results.setdefault("LogisticRegression", []).append({"fold": row["fold"], "accuracy": acc_lr, "roc_auc": auc_lr})
    
    # Random Forest (raw)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    y_proba_rf = rf.predict_proba(X_test)[:, 1]
    acc_rf = accuracy_score(y_test, y_pred_rf)
    auc_rf = roc_auc_score(y_test, y_proba_rf)
    results.setdefault("RandomForest", []).append({"fold": row["fold"], "accuracy": acc_rf, "roc_auc": auc_rf})
    
    # LightGBM (raw)
    lgb_model = lgb.LGBMClassifier(random_state=42)
    lgb_model.fit(X_train, y_train)
    y_pred_lgb = lgb_model.predict(X_test)
    y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]
    acc_lgb = accuracy_score(y_test, y_pred_lgb)
    auc_lgb = roc_auc_score(y_test, y_proba_lgb)
    results.setdefault("LightGBM", []).append({"fold": row["fold"], "accuracy": acc_lgb, "roc_auc": auc_lgb})
    
    # XGBoost (raw)
    xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    acc_xgb = accuracy_score(y_test, y_pred_xgb)
    auc_xgb = roc_auc_score(y_test, y_proba_xgb)
    results.setdefault("XGBoost", []).append({"fold": row["fold"], "accuracy": acc_xgb, "roc_auc": auc_xgb})

# Backtest on Philippines (all dates)
backtest_mask = df["Country"] == backtest_country
X_train_all = df.loc[~backtest_mask, feature_cols].fillna(0)
y_train_all = df.loc[~backtest_mask, target]
X_backtest = df.loc[backtest_mask, feature_cols].fillna(0)
y_backtest = df.loc[backtest_mask, target]

# Scaled features only for Logistic Regression backtest
scaler = StandardScaler()
X_train_all_scaled = scaler.fit_transform(X_train_all)
X_backtest_scaled = scaler.transform(X_backtest)

# Logistic Regression backtest (scaled)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_all_scaled, y_train_all)
y_pred_lr = lr.predict(X_backtest_scaled)
y_proba_lr = lr.predict_proba(X_backtest_scaled)[:, 1]
acc_lr = accuracy_score(y_backtest, y_pred_lr)
auc_lr = roc_auc_score(y_backtest, y_proba_lr)
results["LogisticRegression_backtest"] = {"accuracy": acc_lr, "roc_auc": auc_lr}

# Random Forest backtest (raw)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_all, y_train_all)
y_pred_rf = rf.predict(X_backtest)
y_proba_rf = rf.predict_proba(X_backtest)[:, 1]
acc_rf = accuracy_score(y_backtest, y_pred_rf)
auc_rf = roc_auc_score(y_backtest, y_proba_rf)
results["RandomForest_backtest"] = {"accuracy": acc_rf, "roc_auc": auc_rf}

# LightGBM backtest (raw)
lgb_model = lgb.LGBMClassifier(random_state=42)
lgb_model.fit(X_train_all, y_train_all)
y_pred_lgb = lgb_model.predict(X_backtest)
y_proba_lgb = lgb_model.predict_proba(X_backtest)[:, 1]
acc_lgb = accuracy_score(y_backtest, y_pred_lgb)
auc_lgb = roc_auc_score(y_backtest, y_proba_lgb)
results["LightGBM_backtest"] = {"accuracy": acc_lgb, "roc_auc": auc_lgb}

# XGBoost backtest (raw)
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_all, y_train_all)
y_pred_xgb = xgb_model.predict(X_backtest)
y_proba_xgb = xgb_model.predict_proba(X_backtest)[:, 1]
acc_xgb = accuracy_score(y_backtest, y_pred_xgb)
auc_xgb = roc_auc_score(y_backtest, y_proba_xgb)
results["XGBoost_backtest"] = {"accuracy": acc_xgb, "roc_auc": auc_xgb}

print("Results for crisis_in_6m:")
for model in ["LogisticRegression", "RandomForest", "LightGBM", "XGBoost"]:
    for fold_result in results[model]:
        print(f"{model} Fold {fold_result['fold']}: Accuracy={fold_result['accuracy']:.3f}, ROC-AUC={fold_result['roc_auc']:.3f}")
    backtest = results[model + "_backtest"]
    print(f"Backtest ({backtest_country}): Accuracy={backtest['accuracy']:.3f}, ROC-AUC={backtest['roc_auc']:.3f}\n")

In [ ]:
# Step 4: Hyperparameter optimisation with Optuna for crisis_in_6m
import optuna

target = "crisis_in_6m"
N_TRIALS = 30  # you can increase this for a more thorough search

def _get_fold_data(row):
    train_mask = (df.index >= row["train_start"]) & (df.index <= row["train_end"]) & (df["Country"] != backtest_country)
    test_mask = (df.index >= row["test_start"]) & (df.index <= row["test_end"]) & (df["Country"] != backtest_country)
    X_train = df.loc[train_mask, feature_cols].fillna(0)
    y_train = df.loc[train_mask, target]
    X_test = df.loc[test_mask, feature_cols].fillna(0)
    y_test = df.loc[test_mask, target]
    return X_train, y_train, X_test, y_test

def objective_logreg(trial):
    C = trial.suggest_float("C", 1e-3, 1e3, log=True)
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
    aucs = []
    for _, row in cv_plan.iterrows():
        X_train, y_train, X_test, y_test = _get_fold_data(row)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        model = LogisticRegression(max_iter=1000, C=C, solver=solver)
        model.fit(X_train_scaled, y_train)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        aucs.append(roc_auc_score(y_test, y_proba))
    return np.mean(aucs)

def objective_rf(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2"])
    aucs = []
    for _, row in cv_plan.iterrows():
        X_train, y_train, X_test, y_test = _get_fold_data(row)
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, y_proba))
    return np.mean(aucs)

def objective_lgb(trial):
    num_leaves = trial.suggest_int("num_leaves", 20, 150)
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    aucs = []
    for _, row in cv_plan.iterrows():
        X_train, y_train, X_test, y_test = _get_fold_data(row)
        model = lgb.LGBMClassifier(
            random_state=42,
            num_leaves=num_leaves,
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            max_depth=max_depth,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, y_proba))
    return np.mean(aucs)

def objective_xgb(trial):
    max_depth = trial.suggest_int("max_depth", 3, 10)
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    aucs = []
    for _, row in cv_plan.iterrows():
        X_train, y_train, X_test, y_test = _get_fold_data(row)
        model = xgb.XGBClassifier(
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            max_depth=max_depth,
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, y_proba))
    return np.mean(aucs)

studies = {}
for name, objective in [
    ("LogisticRegression", objective_logreg),
    ("RandomForest", objective_rf),
    ("LightGBM", objective_lgb),
    ("XGBoost", objective_xgb),
]:
    print(f"Optimising {name}...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    studies[name] = study
    print(f"Best AUC for {name}: {study.best_value:.3f}")
    print(f"Best params for {name}: {study.best_params}\n")